# 03 — Bouncing

In notebook 02, we gave the ball gravity. It fell. It hit the floor. It stopped.

Unsatisfying.

Add one line when the ball touches the floor:

```python
dy = -dy
```

Run it. The ball bounces.

This notebook explains what that one line actually did — mathematically, geometrically, and as a programming pattern you already know by a different name.

**Prerequisites:** [02 — Acceleration](./02-acceleration.ipynb). The quadratic motion formula $y = y_0 + v_0 t + \frac{1}{2}gt^2$.

---

## 1. One Line Brings It Back

In notebook 02, the game loop ended like this:

```python
dy += gravity
y  += dy
if y <= floor_y:
    y = floor_y   # stop at the floor, nothing else
```

Add one line at the boundary:

```python
if y <= floor_y:
    y  = floor_y
    dy = -dy      # ← this
```

The ball now returns from where it came. Watch the `dy` label: when the ball hits the floor, the sign flips but the magnitude stays the same.

In [ ]:
# FuncAnimation requires an interactive backend — add '%matplotlib widget' above if it renders static
import matplotlib.pyplot as plt
import matplotlib.animation as animation

x0, y0   = 5.0, 26.0
dx       = 0.8
gravity  = -0.15
floor_y  = 4.0
FRAMES   = 150

# Pre-simulate the full trajectory so bounce positions are known before drawing
x, y, dy_cur = x0, y0, 0.0
all_positions = []
bounce_events = []   # (x_pos, frame_number)

for f in range(FRAMES):
    dy_cur += gravity
    y      += dy_cur
    x      += dx
    if y <= floor_y:
        y      = floor_y
        dy_cur = -dy_cur   # THE LINE
        bounce_events.append((x, f))
    all_positions.append((x, y, dy_cur))

# --- Build the figure ---
fig, ax = plt.subplots(figsize=(10, 3))
fig.patch.set_facecolor('#1e1e2e')
ax.set_facecolor('#1e1e2e')
ax.set_xlim(0, 120)
ax.set_ylim(0, 30)
ax.set_aspect('equal')
ax.axis('off')

ax.axhline(floor_y, color='#45475a', linewidth=1.5)
ax.text(1, floor_y + 0.7, 'floor', color='#6c7086', fontsize=8, fontfamily='monospace')

# Static bounce markers — drawn once before animation starts
for i, (bx, _) in enumerate(bounce_events, 1):
    ax.plot(bx, floor_y, 'v', color='#f38ba8', markersize=7, zorder=5)
    ax.text(bx, floor_y - 2.5, f'#{i}', color='#f38ba8',
            fontsize=8, ha='center', fontfamily='monospace')

ball = plt.Circle((x0, y0), radius=3, color='#89dceb')
ax.add_patch(ball)
trail_dots = [plt.Circle((0, 0), radius=0.8, color='#89dceb', alpha=0) for _ in range(22)]
for dot in trail_dots:
    ax.add_patch(dot)

info = ax.text(2, 28.5, '', color='#cdd6f4', fontsize=9, fontfamily='monospace')

def update(frame):
    px, py, pdy = all_positions[frame]
    ball.set_center((px, py))
    for i, dot in enumerate(trail_dots):
        idx = frame - len(trail_dots) + i + 1
        if 0 <= idx < len(all_positions):
            dot.set_center(all_positions[idx][:2])
            dot.set_alpha(0.04 + 0.045 * i)
        else:
            dot.set_alpha(0)
    info.set_text(f't={frame:>3}   dy={pdy:+.2f}   y={py:.1f}')
    return [ball, info] + trail_dots

ani = animation.FuncAnimation(fig, update, frames=FRAMES, interval=60, blit=True)
plt.tight_layout()
plt.show()

---

## 2. The Flip — Reflection

`dy = -dy` negates the vertical velocity. That is a precise geometric operation: **reflection about a horizontal axis**.

A vector pointing downward with magnitude $v$ becomes a vector pointing upward with magnitude $v$. Same length, opposite direction.

$$v_y \;\longrightarrow\; -v_y$$

In geometry, reflecting a point $(x, y)$ about the x-axis produces $(x, -y)$. The same rule applies to vectors: reflecting the velocity vector $(v_x, v_y)$ about a horizontal axis produces $(v_x, -v_y)$.

Notice what does **not** change: the horizontal velocity `dx` is untouched. The floor is a horizontal boundary — only the component perpendicular to it reverses. The component parallel to it (`dx`) carries on unchanged.

This is not a trick or a shortcut. It is the precise mathematical definition of an elastic reflection.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
fig.patch.set_facecolor('#1e1e2e')
ax.set_facecolor('#1e1e2e')
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.0, 3.5)
ax.set_aspect('equal')
ax.axis('off')

# Floor
ax.axhline(0, color='#45475a', linewidth=2.0, xmin=0.05, xmax=0.95)
ax.text(0, -0.45, 'floor', color='#6c7086', ha='center', fontsize=10, fontfamily='monospace')

# Incoming velocity arrow — pointing downward on the left
ax.annotate('', xy=(-1.0, 0.1), xytext=(-1.0, 2.5),
            arrowprops=dict(arrowstyle='->', color='#f38ba8', lw=2.5))
ax.text(-2.4, 1.3, 'dy = −v\n(falling)', color='#f38ba8',
        ha='center', fontsize=10, fontweight='bold')

# Outgoing velocity arrow — pointing upward on the right
ax.annotate('', xy=(1.0, 2.5), xytext=(1.0, 0.1),
            arrowprops=dict(arrowstyle='->', color='#a6e3a1', lw=2.5))
ax.text(2.4, 1.3, 'dy = −(−v) = v\n(rising)', color='#a6e3a1',
        ha='center', fontsize=10, fontweight='bold')

# Magnitude brackets
for side, col in [(-0.4, '#585b70'), (0.4, '#585b70')]:
    ax.annotate('', xy=(side, 0.1), xytext=(side, 2.5),
                arrowprops=dict(arrowstyle='<->', color=col, lw=1.2))
ax.text(0, 1.3, '|v|', color='#585b70', ha='center', fontsize=9)

# The operation
ax.text(0, 3.1, 'dy = −dy', color='#cdd6f4', ha='center', fontsize=13,
        fontfamily='monospace', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#313244',
                  edgecolor='#585b70', alpha=0.95))

plt.title('Reflection about a horizontal axis', color='#cdd6f4', pad=10, fontsize=11)
plt.tight_layout()
plt.show()

# Show numerically: same magnitude, opposite sign
print('Before bounce:')
for dy_before in [-3.0, -5.5, -8.2]:
    dy_after = -dy_before
    print(f'  dy = {dy_before:+.1f}  →  dy = -dy = {dy_after:+.1f}'
          f'   |before| = {abs(dy_before):.1f},  |after| = {abs(dy_after):.1f}  ✓')

---

## 3. Piecewise Functions

The full trajectory is not one quadratic equation. Each bounce resets the starting conditions — a new arc begins with a new $y_0$, a new $v_0$, the same gravity. The complete trajectory is a sequence of arcs **stitched together at the bounce points**.

This is a **piecewise function**: different rules apply in different regions of the domain.

Using $y_0 = 36$, $v_0 = 0$, $g = -2$, floor at $y = 0$:

- Arc 0 ends at $t = 6$: impact velocity $= -12$, reflected velocity $= 12$
- Arc 1 ends at $t = 18$: same impact speed, same reflected velocity
- Each arc is identical in shape — same height, same duration

$$y(t) = \begin{cases}
36 - t^2 & 0 \leq t < 6 \\[4pt]
12(t - 6) - (t - 6)^2 & 6 \leq t < 18 \\[4pt]
12(t - 18) - (t - 18)^2 & 18 \leq t < 30 \\[4pt]
\vdots
\end{cases}$$

Programmers know this pattern already. It is an `if / elif`:

```python
if 0 <= t < 6:
    y = 36 - t**2
elif 6 <= t < 18:
    τ = t - 6
    y = 12*τ - τ**2
elif 18 <= t < 30:
    τ = t - 18
    y = 12*τ - τ**2
```

The mathematician's piecewise brace and the programmer's `if/elif` are the same concept — different rules for different regions of the input. The notation differs; the structure does not.

The game loop implements this piecewise function dynamically. It does not pre-compute which arc we are in — it just integrates forward and applies the boundary rule when hit. The output is the same.

In [ ]:
import math

def arc(y0, v0, g, tau):
    """Position on a single parabolic arc, tau seconds after arc start."""
    return y0 + v0 * tau + 0.5 * g * tau**2

def bounce_analytic(t, y0=36.0, v0=0.0, g=-2.0, floor=0.0):
    """
    Exact position of a bouncing ball at time t.
    Implements the piecewise function: each arc computed from its own starting conditions.
    """
    y, v = float(y0), float(v0)
    elapsed = 0.0
    while True:
        # Time until this arc hits the floor: solve y + v*τ + ½g*τ² = floor
        a, b, c = 0.5 * g, v, y - floor
        disc = b**2 - 4 * a * c
        sq = math.sqrt(max(disc, 0.0))
        roots = [(-b + sq) / (2*a), (-b - sq) / (2*a)]
        t_arc = min(r for r in roots if r > 1e-9)   # smallest positive root
        if t - elapsed <= t_arc:
            return arc(y, v, g, t - elapsed)
        # Advance to the bounce
        v_impact = v + g * t_arc
        elapsed += t_arc
        y, v = floor, -v_impact   # reset: floor, reflected velocity

def bounce_simulation(t_max, y0=36.0, v0=0.0, g=-2.0, floor=0.0):
    """Game-loop simulation for comparison."""
    dy, y = float(v0), float(y0)
    positions = []
    for _ in range(int(t_max)):
        dy += g
        y  += dy
        if y <= floor:
            y  = floor
            dy = -dy
        positions.append(y)
    return positions

# Compare analytic vs simulation at key moments
sim = bounce_simulation(35)

print(f'{"t":>5}  {"analytic":>10}  {"simulation":>12}  {"diff":>8}')
print('-' * 42)
for t in [1, 3, 5, 6, 9, 12, 15, 18, 21, 24, 27, 30]:
    analytic = bounce_analytic(t)
    sim_val  = sim[t - 1] if t <= len(sim) else float('nan')
    diff     = abs(analytic - sim_val)
    print(f'{t:>5}  {analytic:>10.3f}  {sim_val:>12.3f}  {diff:>8.3f}')

print()
print('Analytic formula: exact. Simulation: Euler approximation (small drift at bounce points).')

---

## 4. The Boundary Condition

The line `if y <= floor_y: dy = -dy` has a formal name in physics: a **boundary condition**.

A differential equation describes how a system evolves. But it does not say what happens at the edges of the domain — the walls, the floor, the boundary. That is supplied separately, as a boundary condition. Together, the equation and its boundary condition fully determine the solution.

Our boundary condition is:

> At $y = y_{\text{floor}}$, the normal component of velocity reverses: $v_y \to -v_y$.

This is called an **elastic** (or **reflective**) boundary condition — no energy is absorbed, the magnitude is conserved. Other boundary conditions are possible:

| Boundary condition | Physics meaning | Code equivalent |
|---|---|---|
| Elastic / reflective | velocity reverses, energy conserved | `dy = -dy` |
| Absorbing / inelastic | object stops at boundary | `dy = 0` |
| Damped | object loses some energy each bounce | `dy = -dy * 0.85` |
| Periodic | object wraps to the opposite edge | `y = ceiling_y` |

Each row is a different `if` block. The choice of boundary condition changes the physical behaviour completely — same equation, different rules at the edges.

The programmer's `if` statement is a boundary condition. This is not a metaphor — they are the same concept in different notation.

In [ ]:
import matplotlib.pyplot as plt

# Same physics, four different boundary conditions
def simulate_bc(y0, dy0, g, floor, ceiling, frames, bc):
    """bc: 'elastic', 'absorbing', 'damped', or 'periodic'"""
    dy, y = float(dy0), float(y0)
    ys = [y]
    for _ in range(frames):
        dy += g
        y  += dy
        if y <= floor:
            if bc == 'elastic':   y = floor;    dy = -dy
            elif bc == 'absorbing': y = floor;  dy = 0.0
            elif bc == 'damped':  y = floor;    dy = -dy * 0.75
            elif bc == 'periodic': y = ceiling
        ys.append(y)
    return ys

y0, dy0, g, floor, ceiling, frames = 30.0, 0.0, -1.0, 0.0, 30.0, 60

configs = [
    ('elastic',   '#a6e3a1', 'Elastic: dy = −dy      (energy conserved)'),
    ('damped',    '#fab387', 'Damped:  dy = −dy × 0.75 (energy lost each bounce)'),
    ('absorbing', '#f38ba8', 'Absorbing: dy = 0        (ball stops at floor)'),
    ('periodic',  '#89dceb', 'Periodic: y = ceiling    (ball wraps)'),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 5))
fig.patch.set_facecolor('#1e1e2e')
axes = axes.flatten()

for ax, (bc, color, label) in zip(axes, configs):
    ys = simulate_bc(y0, dy0, g, floor, ceiling, frames, bc)
    ax.set_facecolor('#1e1e2e')
    ax.tick_params(colors='#cdd6f4', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#45475a')
    ax.axhline(floor,   color='#45475a', linewidth=1.0, linestyle='-')
    ax.axhline(ceiling, color='#45475a', linewidth=0.6, linestyle=':')
    ax.plot(ys, color=color, linewidth=1.8)
    ax.set_xlim(0, frames)
    ax.set_ylim(-2, 34)
    ax.set_xlabel('t', color='#cdd6f4', fontsize=9)
    ax.set_ylabel('y', color='#cdd6f4', fontsize=9)
    ax.set_title(label, color='#cdd6f4', fontsize=9, pad=6)

plt.suptitle('Same equation, four boundary conditions', color='#cdd6f4', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---

## 5. Plot the Trajectory

Plot `y` against `t` for the bouncing ball. The piecewise nature is unmistakable: a series of parabolic arcs, each a mirror of the last, stitched together where they touch the floor.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Use the analytic formula for clean, exact arcs
t_vals  = np.linspace(0, 30, 1200)
y_vals  = np.array([bounce_analytic(t) for t in t_vals])

# Bounce times for this setup (y0=36, g=-2): t=6, 18, 30, ...
bounce_times = [6, 18, 30]

fig, ax = plt.subplots(figsize=(11, 4))
fig.patch.set_facecolor('#1e1e2e')
ax.set_facecolor('#1e1e2e')
ax.tick_params(colors='#cdd6f4')
for spine in ax.spines.values():
    spine.set_edgecolor('#45475a')

ax.plot(t_vals, y_vals, color='#89dceb', linewidth=2.2)
ax.axhline(0, color='#45475a', linewidth=1.5)
ax.axhline(36, color='#45475a', linewidth=0.6, linestyle=':', alpha=0.5)
ax.text(0.3, 37, 'starting height = 36', color='#6c7086', fontsize=9)

# Mark and label the bounce points
for i, tb in enumerate(bounce_times, 1):
    ax.axvline(tb, color='#f38ba8', linewidth=0.8, linestyle='--', alpha=0.6)
    ax.plot(tb, 0, 'o', color='#f38ba8', markersize=7, zorder=5)
    ax.text(tb, -3.5, f'bounce {i}\nt = {tb}', color='#f38ba8',
            ha='center', fontsize=8, fontfamily='monospace')

# Label the arcs
arc_midpoints = [(3, 20), (12, 20), (24, 20)]
arc_labels    = ['arc 0\n(free fall)', 'arc 1', 'arc 2']
for (tx, ty), label in zip(arc_midpoints, arc_labels):
    ax.text(tx, ty, label, color='#cdd6f4', ha='center', fontsize=9, alpha=0.7)

ax.set_xlabel('t  (time)', color='#cdd6f4')
ax.set_ylabel('y  (position)', color='#cdd6f4')
ax.set_title('Bouncing ball trajectory — a piecewise function', color='#cdd6f4', pad=12)
ax.set_xlim(0, 30)
ax.set_ylim(-6, 42)
plt.tight_layout()
plt.show()

print('Each arc is identical in height (36) and duration.')
print('Arc 0 lasts 6 time units (free fall from rest).')
print('Arcs 1, 2, ... each last 12 time units (same impact speed, symmetric rise and fall).')
print()
print('Arc durations:')
durations = [6, 18-6, 30-18]
for i, d in enumerate(durations):
    print(f'  Arc {i}: {d} time units')

---

## 6. Worked Examples

Tracing two full bounces step by step, using $y_0 = 36$, $v_0 = 0$, $g = -2$, floor at $y = 0$.

### Arc 0 — Free fall from rest

$$y(t) = 36 + 0 \cdot t + \tfrac{1}{2}(-2) t^2 = 36 - t^2$$

Floor hit: $36 - t^2 = 0 \Rightarrow t = 6$

Impact velocity: $v(6) = v_0 + g \cdot t = 0 + (-2)(6) = -12$

Apply reflection: $v \to -v = 12$ (upward)

Peak during arc 0: $y(3) = 36 - 9 = 27$. No — wait: arc 0 is a drop, so the maximum is at the start: $y(0) = 36$.

### Arc 1 — First bounce

New starting conditions: $y_1 = 0$, $v_1 = 12$, same $g = -2$. Let $\tau = t - 6$:

$$y(\tau) = 0 + 12\tau + \tfrac{1}{2}(-2)\tau^2 = 12\tau - \tau^2$$

Peak: $v = 12 - 2\tau = 0 \Rightarrow \tau = 6 \Rightarrow y(6) = 72 - 36 = \mathbf{36}$ — same height as the start.

Floor hit: $12\tau - \tau^2 = 0 \Rightarrow \tau(12 - \tau) = 0 \Rightarrow \tau = 12$, so $t = 18$.

Impact velocity: $v(12) = 12 + (-2)(12) = 12 - 24 = -12$

Apply reflection: $v \to 12$ — identical to the start of arc 1.

### Arc 2 — Second bounce

Same starting conditions as arc 1. The trajectory repeats exactly — same height, same duration. This continues indefinitely: **perfect elastic bounce, no energy loss**.

In [ ]:
import math

def arc_y(y0, v0, g, tau):
    return y0 + v0 * tau + 0.5 * g * tau**2

def arc_v(v0, g, tau):
    return v0 + g * tau

# --- Arc 0 ---
y0, v0, g = 36.0, 0.0, -2.0

print('Arc 0 — free fall from y=36')
print(f'  Formula:  y(t) = {y0} + {v0}t + ½({g})t² = {y0} − t²')
for t in [0, 3, 5, 6]:
    print(f'  t={t}: y = {arc_y(y0, v0, g, t):.1f}  v = {arc_v(v0, g, t):.1f}')
t_impact_0 = math.sqrt(-y0 / (0.5 * g))   # solve 36 - t² = 0
v_impact_0 = arc_v(v0, g, t_impact_0)
print(f'  Floor hit at t = sqrt({y0}) = {t_impact_0:.4f}')
print(f'  Impact velocity: {v_impact_0:.4f}')
print(f'  After reflection: {-v_impact_0:.4f}')
print()

# --- Arc 1 ---
y1, v1 = 0.0, -v_impact_0   # = 12

print('Arc 1 — first bounce')
print(f'  Starting: y={y1}, v={v1:.4f}, g={g}')
print(f'  Formula:  y(τ) = {v1:.0f}τ − τ²  (τ = t − {t_impact_0:.0f})')
for tau in [0, 3, 6, 9, 12]:
    print(f'  τ={tau:>2}: y = {arc_y(y1, v1, g, tau):.1f}  v = {arc_v(v1, g, tau):.1f}')
t_peak_1 = -v1 / g
y_peak_1 = arc_y(y1, v1, g, t_peak_1)
t_floor_1 = -v1 / (0.5 * g)   # solve 12τ - τ² = 0, non-zero root
v_impact_1 = arc_v(v1, g, t_floor_1)
print(f'  Peak at τ={t_peak_1:.0f}: y = {y_peak_1:.1f}  (same as starting height ✓)')
print(f'  Floor hit at τ={t_floor_1:.0f} (t={t_impact_0:.0f}+{t_floor_1:.0f}={t_impact_0+t_floor_1:.0f})')
print(f'  Impact velocity: {v_impact_1:.4f}')
print(f'  After reflection: {-v_impact_1:.4f}  (same as arc 1 start ✓ — repeats forever)')
print()

# Verify against simulation
print('Simulation vs formula at key points:')
sim = bounce_simulation(30)
check_points = [(5, 'just before floor'), (6, 'floor (arc 0 end)'),
                (12, 'peak arc 1'), (18, 'floor (arc 1 end)'), (24, 'peak arc 2')]
for t, label in check_points:
    formula = bounce_analytic(t)
    sim_val = sim[t - 1]
    print(f'  t={t:>2} ({label}):  formula={formula:6.2f},  sim={sim_val:6.2f}')

---

## Exercises

- **Set `dy = 0` instead of `dy = -dy`** at the floor. What boundary condition is that? What does the trajectory plot look like?
- **Set `dy = -dy * 0.85`** at the floor. This is a damped boundary condition — the ball loses 15% of its speed on each bounce. Watch the arcs shrink. Can you predict, from the formula, how many bounces it takes before the ball no longer clears a height of 5?
- **Add a ceiling** at `y = ceiling_y`: if `y >= ceiling_y: dy = -dy`. The ball now bounces off both floor and ceiling. What does the trajectory plot look like now?
- **Add side walls**: `if x <= 0 or x >= screen_width: dx = -dx`. The ball now bounces in 2D. Think about this: reflecting off a horizontal floor negates `dy`. Reflecting off a vertical wall negates `dx`. What is the general rule? Which component of velocity reverses depends on which axis the wall is parallel to. Notebook 04 will build on this.

---

## Summary

| Concept | Plain English | Code | Math |
|---------|--------------|------|------|
| Reflection | same speed, opposite direction | `dy = -dy` | $v_y \to -v_y$ |
| Elastic bounce | no energy lost at boundary | `\|dy\|` unchanged | $\|v_y^{\text{after}}\| = \|v_y^{\text{before}}\|$ |
| Piecewise function | different rules in different regions | `if / elif` blocks | $f(x) = \begin{cases} f_1(x) & x < x_1 \\ f_2(x) & x \geq x_1 \end{cases}$ |
| Boundary condition | what happens at the edge of the domain | the `if` at the floor | constraint on the ODE solution |
| Arc stitching | each bounce starts a new parabolic arc | reset $y_0$, $v_0$ at bounce | piecewise quadratic |

**The through-line:** `dy = -dy` is not a hack — it is a geometric reflection, a boundary condition, and a piecewise rule, all at once. Each of those names describes the same one-line operation from a different angle.

The programmer already knew how to write it. The math gives it a name and connects it to a broader family of techniques.

---

**Next:** [04 — 2D Motion](./04-2d-motion.ipynb) — the ball gains a second dimension. Velocity becomes a vector, and reflection generalises to arbitrary walls.